# Zone Start Signal Analysis (Phase 2, Area 2)

**Goal:** Determine whether `faceoff_zone_code` (already in `shot_events`) captures sufficient zone-deployment context for the xG model, or whether full shift data ingestion is needed.

**Key questions:**
1. Does faceoff zone code meaningfully distinguish shot quality within various time windows?
2. Is `seconds_since_faceoff` a useful proxy for on-the-fly vs. post-faceoff context?
3. Is shift data available historically from the NHL API?

**Gate:** If `faceoff_zone_code` alone does NOT distinguish shot quality by zone context, shift ingestion becomes a Phase 2.5 or Phase 4 priority.

**Data source:** `shot_events` table in `data/nhl_data.db`

In [ ]:
import os
import sys
import sqlite3

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats as sp_stats

# Add src/ to path ? handle CWD being project root or notebooks/
for _candidate in [os.path.join(os.getcwd(), "src"),
                   os.path.join(os.getcwd(), "..", "src")]:
    _candidate = os.path.abspath(_candidate)
    if os.path.isdir(_candidate) and _candidate not in sys.path:
        sys.path.insert(0, _candidate)
        break

from database import DATABASE_PATH, BLOCKED_SHOT_EVENT_TYPE
from validation import (
    COHEN_H_SMALL,
    MIN_SHOTS_PER_CELL,
    bootstrap_goal_rate_ci,
    cohens_h,
)

NOTEBOOK_CI_BOOTSTRAPS = 2_000
SIGNIFICANCE_ALPHA = 0.05
ANALYSIS_SHOT_WHERE = f"(shot_event_type IS NULL OR shot_event_type != '{BLOCKED_SHOT_EVENT_TYPE}')"


def p_value_label(p_value):
    if p_value is None:
        return "N/A"
    return "<0.0001" if p_value < 0.0001 else f"{p_value:.4f}"


def analyze_goal_rate_groups(rows, group_field):
    valid_rows = [row for row in rows if row["shots"] and row["shots"] > 0]
    total_shots = sum(row["shots"] for row in valid_rows)
    total_goals = sum(row["goals"] or 0 for row in valid_rows)
    overall_rate = total_goals / total_shots if total_shots else 0.0

    if len(valid_rows) >= 2:
        observed = np.array([
            [row["goals"] or 0, row["shots"] - (row["goals"] or 0)]
            for row in valid_rows
        ])
        chi2, p_value, dof, _ = sp_stats.chi2_contingency(observed)
    else:
        chi2, p_value, dof = None, None, None

    summaries = []
    for row in valid_rows:
        shots = row["shots"]
        goals = row["goals"] or 0
        rate, ci_low, ci_high = bootstrap_goal_rate_ci(
            goals, shots, n_boot=NOTEBOOK_CI_BOOTSTRAPS
        )
        summaries.append({
            "group": row[group_field],
            "shots": shots,
            "goals": goals,
            "rate": rate,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "cohens_h": cohens_h(rate, overall_rate),
            "underpowered": shots < MIN_SHOTS_PER_CELL,
        })

    return {
        "chi2": chi2,
        "p_value": p_value,
        "dof": dof,
        "overall_rate": overall_rate,
        "groups": summaries,
    }


def print_goal_rate_analysis(title, analysis, group_header):
    print(f"\n{title}")
    if analysis["chi2"] is not None:
        print(
            f"Chi-squared: statistic={analysis['chi2']:.2f}, "
            f"dof={analysis['dof']}, p={p_value_label(analysis['p_value'])}"
        )
    else:
        print("Chi-squared: N/A (fewer than two populated groups)")

    print(
        f"{group_header:<18} {'Shots':>9} {'Goals':>7} {'Rate':>8} "
        f"{'95% CI':>17} {'h':>8} {'Power':>7}"
    )
    print("-" * 82)
    for row in analysis["groups"]:
        ci_label = f"[{row['ci_low']*100:.2f}, {row['ci_high']*100:.2f}]"
        power = "LOW" if row["underpowered"] else "OK"
        print(
            f"{str(row['group']):<18} {row['shots']:>9,} {row['goals']:>7,} "
            f"{row['rate']*100:>7.2f}% {ci_label:>17} "
            f"{row['cohens_h']:>+8.3f} {power:>7}"
        )

    meaningful = [
        row for row in analysis["groups"]
        if abs(row["cohens_h"]) >= COHEN_H_SMALL and not row["underpowered"]
    ]
    if meaningful:
        labels = ", ".join(str(row["group"]) for row in meaningful)
        print(f"Practically meaningful groups: {labels}")
    else:
        print("No adequately powered group reaches the practical-effect threshold.")


sns.set_theme(style="whitegrid")
print(f"Database: {DATABASE_PATH}")
conn = sqlite3.connect(DATABASE_PATH)
conn.row_factory = sqlite3.Row
print("Connected.")


## 1. Data availability check

Verify we have `faceoff_zone_code` and `seconds_since_faceoff` populated in `shot_events`.

In [ ]:
cur = conn.cursor()

cur.execute("SELECT COUNT(*) FROM shot_events")
total_shots = cur.fetchone()[0]

cur.execute(f"SELECT COUNT(*) FROM shot_events WHERE {ANALYSIS_SHOT_WHERE}")
analysis_shots = cur.fetchone()[0]

cur.execute(f"SELECT COUNT(*) FROM shot_events WHERE {ANALYSIS_SHOT_WHERE} AND faceoff_zone_code IS NOT NULL")
with_zone = cur.fetchone()[0]

cur.execute(f"SELECT COUNT(*) FROM shot_events WHERE {ANALYSIS_SHOT_WHERE} AND seconds_since_faceoff IS NOT NULL")
with_timing = cur.fetchone()[0]

cur.execute(f"""
    SELECT COUNT(*)
    FROM shot_events
    WHERE {ANALYSIS_SHOT_WHERE}
      AND faceoff_zone_code IS NOT NULL
      AND seconds_since_faceoff IS NOT NULL
""")
with_both = cur.fetchone()[0]

cur.execute(f"""
    SELECT DISTINCT faceoff_zone_code
    FROM shot_events
    WHERE {ANALYSIS_SHOT_WHERE}
      AND faceoff_zone_code IS NOT NULL
""")
zone_codes = [r[0] for r in cur.fetchall()]

print(f"Total shot events:                   {total_shots:,}")
print(f"Analysis-eligible shot events:       {analysis_shots:,}")
print(f"  with faceoff_zone_code:            {with_zone:,} ({with_zone/analysis_shots*100:.1f}%)" if analysis_shots else "")
print(f"  with seconds_since_faceoff:        {with_timing:,} ({with_timing/analysis_shots*100:.1f}%)" if analysis_shots else "")
print(f"  with both:                         {with_both:,} ({with_both/analysis_shots*100:.1f}%)" if analysis_shots else "")
print(f"  distinct zone codes:               {sorted(zone_codes)}")

if with_both == 0:
    print("\n*** No zone + timing data. Run the scraper to populate shot_events. ***")


## 2. Shot quality by faceoff zone code (all shots)

Does the zone of the preceding faceoff distinguish shot quality? Compare goal rate, average distance, and shot count across zone codes (O = offensive, D = defensive, N = neutral).

In [ ]:
cur.execute(f"""
    SELECT faceoff_zone_code,
           COUNT(*) AS shots,
           SUM(is_goal) AS goals,
           CAST(SUM(is_goal) AS REAL) / COUNT(*) AS goal_rate,
           ROUND(AVG(distance_to_goal), 1) AS avg_distance,
           ROUND(AVG(ABS(angle_to_goal)), 1) AS avg_angle
    FROM shot_events
    WHERE {ANALYSIS_SHOT_WHERE}
      AND faceoff_zone_code IS NOT NULL
    GROUP BY faceoff_zone_code
    ORDER BY faceoff_zone_code
""")
zone_all = cur.fetchall()

if zone_all:
    codes = [r["faceoff_zone_code"] for r in zone_all]
    shots = [r["shots"] for r in zone_all]
    rates = [r["goal_rate"] for r in zone_all]
    dists = [r["avg_distance"] for r in zone_all]

    zone_colors = {"D": "#e74c3c", "N": "#f39c12", "O": "#2ecc71"}
    colors = [zone_colors.get(c, "#3498db") for c in codes]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].bar(codes, rates, color=colors, alpha=0.8)
    axes[0].set_ylabel("Goal rate")
    axes[0].set_title("Goal Rate by Faceoff Zone")
    axes[0].set_xlabel("Faceoff zone code")

    axes[1].bar(codes, dists, color=colors, alpha=0.8)
    axes[1].set_ylabel("Average distance to goal (ft)")
    axes[1].set_title("Avg Shot Distance by Faceoff Zone")
    axes[1].set_xlabel("Faceoff zone code")

    axes[2].bar(codes, shots, color=colors, alpha=0.8)
    axes[2].set_ylabel("Number of shots")
    axes[2].set_title("Shot Count by Faceoff Zone")
    axes[2].set_xlabel("Faceoff zone code")

    fig.tight_layout()
    plt.show()

    zone_all_analysis = analyze_goal_rate_groups(zone_all, "faceoff_zone_code")
    print_goal_rate_analysis("Faceoff zone all-shot goal-rate test", zone_all_analysis, "Zone")
    print(f"\n{'Zone':<6} {'Avg Dist':>10} {'Avg Angle':>10}")
    print("-" * 30)
    for row in zone_all:
        print(f"{row['faceoff_zone_code']:<6} {row['avg_distance']:>10.1f} {row['avg_angle']:>10.1f}")
else:
    print("No faceoff zone data available.")


## 3. Zone signal within time windows

The critical test: does `faceoff_zone_code` distinguish shot quality **within narrow time windows** after the faceoff? If OZ faceoffs produce higher-quality shots only in the first 10 seconds, that's a strong signal. If the effect persists at 30+ seconds, it may reflect sustained possession rather than the faceoff itself.

In [ ]:
TIME_WINDOWS = [
    (0, 5, "0-5s"),
    (6, 10, "6-10s"),
    (11, 15, "11-15s"),
    (16, 30, "16-30s"),
    (31, 60, "31-60s"),
    (61, 300, "61+s"),
]

results = []
window_analyses = {}
for lo, hi, label in TIME_WINDOWS:
    cur.execute(f"""
        SELECT faceoff_zone_code,
               COUNT(*) AS shots,
               SUM(is_goal) AS goals,
               CAST(SUM(is_goal) AS REAL) / COUNT(*) AS goal_rate,
               ROUND(AVG(distance_to_goal), 1) AS avg_distance
        FROM shot_events
        WHERE {ANALYSIS_SHOT_WHERE}
          AND faceoff_zone_code IS NOT NULL
          AND seconds_since_faceoff BETWEEN ? AND ?
        GROUP BY faceoff_zone_code
        ORDER BY faceoff_zone_code
    """, (lo, hi))
    rows = cur.fetchall()
    if rows:
        window_analyses[label] = analyze_goal_rate_groups(rows, "faceoff_zone_code")
    for r in rows:
        results.append((label, r["faceoff_zone_code"], r["shots"], r["goals"], r["goal_rate"], r["avg_distance"]))

if results:
    zone_set = sorted(set(r[1] for r in results))
    window_labels = [w[2] for w in TIME_WINDOWS]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

    x = np.arange(len(window_labels))
    bar_width = 0.25

    for i, zone in enumerate(zone_set):
        zone_rates = []
        zone_dists = []
        for wlabel in window_labels:
            match = [r for r in results if r[0] == wlabel and r[1] == zone]
            zone_rates.append(match[0][4] if match else 0)
            zone_dists.append(match[0][5] if match else 0)

        color = zone_colors.get(zone, "#3498db")
        ax1.bar(x + i * bar_width, zone_rates, bar_width, label=f"Zone {zone}",
                color=color, alpha=0.8)
        ax2.bar(x + i * bar_width, zone_dists, bar_width, label=f"Zone {zone}",
                color=color, alpha=0.8)

    ax1.set_xlabel("Time window after faceoff")
    ax1.set_ylabel("Goal rate")
    ax1.set_title("Goal Rate by Faceoff Zone Within Time Windows")
    ax1.set_xticks(x + bar_width)
    ax1.set_xticklabels(window_labels)
    ax1.legend()

    ax2.set_xlabel("Time window after faceoff")
    ax2.set_ylabel("Avg distance to goal (ft)")
    ax2.set_title("Avg Shot Distance by Faceoff Zone Within Time Windows")
    ax2.set_xticks(x + bar_width)
    ax2.set_xticklabels(window_labels)
    ax2.legend()

    fig.tight_layout()
    plt.show()

    for label in window_labels:
        analysis = window_analyses.get(label)
        if analysis:
            print_goal_rate_analysis(f"Window {label}: zone goal-rate test", analysis, "Zone")
else:
    print("No time-windowed zone data available.")


## 4. Seconds-since-faceoff as on-the-fly proxy

Shots that occur long after a faceoff (60+ seconds) likely followed a change on the fly rather than a set play off a faceoff. Do these "steady state" shots look different from post-faceoff shots? If so, `seconds_since_faceoff` captures a useful on-the-fly vs. set-play distinction without needing shift data.

In [ ]:
POST_FACEOFF_WINDOW = 10
STEADY_STATE_THRESHOLD = 61

cur.execute(f"""
    SELECT
        CASE
            WHEN seconds_since_faceoff <= ? THEN 'Post-faceoff (0-10s)'
            WHEN seconds_since_faceoff >= ? THEN 'Steady state (61+s)'
            ELSE 'Transition (11-60s)'
        END AS phase,
        COUNT(*) AS shots,
        SUM(is_goal) AS goals,
        CAST(SUM(is_goal) AS REAL) / COUNT(*) AS goal_rate,
        ROUND(AVG(distance_to_goal), 1) AS avg_distance,
        ROUND(AVG(ABS(angle_to_goal)), 1) AS avg_angle
    FROM shot_events
    WHERE {ANALYSIS_SHOT_WHERE}
      AND seconds_since_faceoff IS NOT NULL
    GROUP BY phase
    ORDER BY phase
""", (POST_FACEOFF_WINDOW, STEADY_STATE_THRESHOLD))
phase_rows = cur.fetchall()

if phase_rows:
    labels = [r["phase"] for r in phase_rows]
    p_shots = [r["shots"] for r in phase_rows]
    p_rates = [r["goal_rate"] for r in phase_rows]
    p_dists = [r["avg_distance"] for r in phase_rows]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    bar_colors = ["#2ecc71", "#f39c12", "#3498db"]

    axes[0].bar(labels, p_rates, color=bar_colors, alpha=0.8)
    axes[0].set_ylabel("Goal rate")
    axes[0].set_title("Goal Rate by Game Phase")
    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=15, ha="right")

    axes[1].bar(labels, p_dists, color=bar_colors, alpha=0.8)
    axes[1].set_ylabel("Avg distance to goal (ft)")
    axes[1].set_title("Avg Shot Distance by Game Phase")
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=15, ha="right")

    axes[2].bar(labels, p_shots, color=bar_colors, alpha=0.8)
    axes[2].set_ylabel("Number of shots")
    axes[2].set_title("Shot Volume by Game Phase")
    plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=15, ha="right")

    fig.tight_layout()
    plt.show()

    phase_analysis = analyze_goal_rate_groups(phase_rows, "phase")
    print_goal_rate_analysis("Seconds-since-faceoff phase goal-rate test", phase_analysis, "Phase")
    print(f"\n{'Phase':<28} {'Avg Dist':>10} {'Avg Angle':>10}")
    print("-" * 52)
    for row in phase_rows:
        print(f"{row['phase']:<28} {row['avg_distance']:>10.1f} {row['avg_angle']:>10.1f}")
else:
    print("No timing data available.")


## 5. Zone-recency interaction heatmap

Cross-tabulate faceoff zone code with time window to reveal where zone signal is strongest. This directly tests whether the interaction feature (`faceoff_zone_recency_interaction`) will be informative.

In [ ]:
# Build a goal-rate heatmap: zone x time window
if results:
    window_order = [w[2] for w in TIME_WINDOWS]
    zone_order = sorted(zone_set)

    rate_matrix = np.zeros((len(zone_order), len(window_order)))
    count_matrix = np.zeros((len(zone_order), len(window_order)), dtype=int)
    power_matrix = np.full((len(zone_order), len(window_order)), "", dtype=object)

    for r in results:
        wi = window_order.index(r[0])
        zi = zone_order.index(r[1])
        rate_matrix[zi, wi] = r[4]
        count_matrix[zi, wi] = r[2]
        power_matrix[zi, wi] = "LOW" if r[2] < MIN_SHOTS_PER_CELL else "OK"

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 4))

    im1 = ax1.imshow(rate_matrix, cmap="RdYlGn", aspect="auto")
    ax1.set_xticks(range(len(window_order)))
    ax1.set_xticklabels(window_order, rotation=30, ha="right")
    ax1.set_yticks(range(len(zone_order)))
    ax1.set_yticklabels([f"Zone {z}" for z in zone_order])
    ax1.set_title("Goal Rate: Zone x Time Window")
    for i in range(len(zone_order)):
        for j in range(len(window_order)):
            ax1.text(j, i, f"{rate_matrix[i, j]:.3f}", ha="center", va="center",
                     fontsize=9, color="black")
    fig.colorbar(im1, ax=ax1, shrink=0.8)

    im2 = ax2.imshow(count_matrix, cmap="Blues", aspect="auto")
    ax2.set_xticks(range(len(window_order)))
    ax2.set_xticklabels(window_order, rotation=30, ha="right")
    ax2.set_yticks(range(len(zone_order)))
    ax2.set_yticklabels([f"Zone {z}" for z in zone_order])
    ax2.set_title("Shot Count: Zone x Time Window")
    for i in range(len(zone_order)):
        for j in range(len(window_order)):
            label = f"{count_matrix[i, j]:,}\n{power_matrix[i, j]}"
            ax2.text(j, i, label, ha="center", va="center",
                     fontsize=8, color="black" if count_matrix[i, j] < count_matrix.max() * 0.7 else "white")
    fig.colorbar(im2, ax=ax2, shrink=0.8)

    fig.tight_layout()
    plt.show()

    print("\nGoal-rate spread and formal-test status by time window:")
    for j, wlabel in enumerate(window_order):
        col = rate_matrix[:, j]
        spread = col.max() - col.min()
        analysis = window_analyses.get(wlabel)
        p_label = p_value_label(analysis["p_value"]) if analysis else "N/A"
        meaningful = bool(analysis and any(
            abs(row["cohens_h"]) >= COHEN_H_SMALL and not row["underpowered"]
            for row in analysis["groups"]
        ))
        print(
            f"  {wlabel:<10}: spread={spread:.4f}, p={p_label}, "
            f"practical_effect={'YES' if meaningful else 'NO'}"
        )
    print("\nA window supports feature screening only when the test is significant, effects are practical, and cells are powered.")
else:
    print("No cross-tab data available (section 3 produced no results).")


## 6. Manpower state interaction

Does the zone signal differ between even-strength and power-play situations? OZ faceoffs on the PP may produce qualitatively different shots than OZ faceoffs at even strength.

In [ ]:
cur.execute(f"""
    SELECT manpower_state,
           faceoff_zone_code,
           COUNT(*) AS shots,
           SUM(is_goal) AS goals,
           CAST(SUM(is_goal) AS REAL) / COUNT(*) AS goal_rate,
           ROUND(AVG(distance_to_goal), 1) AS avg_distance
    FROM shot_events
    WHERE {ANALYSIS_SHOT_WHERE}
      AND faceoff_zone_code IS NOT NULL
      AND manpower_state IS NOT NULL
      AND seconds_since_faceoff <= ?
    GROUP BY manpower_state, faceoff_zone_code
    ORDER BY manpower_state, faceoff_zone_code
""", (POST_FACEOFF_WINDOW,))
mp_rows = cur.fetchall()

if mp_rows:
    mp_states = sorted(set(r["manpower_state"] for r in mp_rows))
    zone_order_mp = sorted(set(r["faceoff_zone_code"] for r in mp_rows))

    print(f"Post-faceoff shots (0-{POST_FACEOFF_WINDOW}s) by manpower state and zone:\n")
    print(f"{'Manpower':<12} {'Zone':<6} {'Shots':>8} {'Goals':>8} {'Goal Rate':>10} {'Avg Dist':>10} {'Power':>7}")
    print("-" * 72)
    for row in mp_rows:
        power = "LOW" if row["shots"] < MIN_SHOTS_PER_CELL else "OK"
        print(
            f"{row['manpower_state']:<12} {row['faceoff_zone_code']:<6} "
            f"{row['shots']:>8,} {row['goals']:>8,} {row['goal_rate']:>10.4f} "
            f"{row['avg_distance']:>10.1f} {power:>7}"
        )

    for mp_state in mp_states:
        rows_for_state = [row for row in mp_rows if row["manpower_state"] == mp_state]
        analysis = analyze_goal_rate_groups(rows_for_state, "faceoff_zone_code")
        print_goal_rate_analysis(f"Manpower {mp_state}: zone goal-rate test", analysis, "Zone")

    mp_rate_matrix = np.zeros((len(mp_states), len(zone_order_mp)))
    for row in mp_rows:
        mi = mp_states.index(row["manpower_state"])
        zi = zone_order_mp.index(row["faceoff_zone_code"])
        mp_rate_matrix[mi, zi] = row["goal_rate"]

    fig, ax = plt.subplots(figsize=(8, max(4, len(mp_states) * 0.8)))
    im = ax.imshow(mp_rate_matrix, cmap="RdYlGn", aspect="auto")
    ax.set_xticks(range(len(zone_order_mp)))
    ax.set_xticklabels([f"Zone {z}" for z in zone_order_mp])
    ax.set_yticks(range(len(mp_states)))
    ax.set_yticklabels(mp_states)
    ax.set_title(f"Goal Rate: Manpower x Zone (first {POST_FACEOFF_WINDOW}s after faceoff)")
    for i in range(len(mp_states)):
        for j in range(len(zone_order_mp)):
            ax.text(j, i, f"{mp_rate_matrix[i, j]:.3f}", ha="center", va="center",
                    fontsize=10, color="black")
    fig.colorbar(im, ax=ax, shrink=0.8)
    fig.tight_layout()
    plt.show()
else:
    print("No manpower x zone data available.")


## 7. Shift data API availability check

Spot-check whether the NHL shift chart API returns data for games across different eras. This informs whether a future shift data pipeline is feasible for the full historical range.

In [ ]:
import requests

SHIFT_API_URL = "https://api.nhle.com/stats/rest/en/shiftcharts"

SAMPLE_GAMES = {}

cur.execute("""
    SELECT game_id, season FROM games
    WHERE season IS NOT NULL
    GROUP BY season
    ORDER BY season
""")
season_samples = cur.fetchall()

if season_samples:
    test_seasons = []
    for r in season_samples:
        season_str = str(r[1])
        if season_str in ("20072008", "20122013", "20172018", "20202021", "20232024", "20242025"):
            SAMPLE_GAMES[r[0]] = season_str
            test_seasons.append(season_str)

    if not SAMPLE_GAMES and len(season_samples) >= 3:
        picks = [season_samples[0], season_samples[len(season_samples) // 2], season_samples[-1]]
        for r in picks:
            SAMPLE_GAMES[r[0]] = str(r[1])

print(f"Testing shift API for {len(SAMPLE_GAMES)} sample games...")

shift_results = []
for game_id, season in sorted(SAMPLE_GAMES.items(), key=lambda x: x[1]):
    try:
        resp = requests.get(
            SHIFT_API_URL,
            params={"cayenneExp": f"gameId={game_id}"},
            timeout=10
        )
        if resp.status_code == 200:
            data = resp.json()
            n_shifts = len(data.get("data", []))
            shift_results.append((game_id, season, "OK", n_shifts))
        else:
            shift_results.append((game_id, season, f"HTTP {resp.status_code}", 0))
    except Exception as e:
        shift_results.append((game_id, season, f"Error: {e}", 0))

print(f"\n{'Game ID':<12} {'Season':<12} {'Status':<20} {'Shifts':>8}")
print("-" * 56)
for gid, season, status, n in shift_results:
    print(f"{gid:<12} {season:<12} {status:<20} {n:>8}")

ok_count = sum(1 for r in shift_results if r[2] == "OK")
with_data = sum(1 for r in shift_results if r[3] > 0)
print(f"\nAPI responded: {ok_count}/{len(shift_results)}")
print(f"With shift data: {with_data}/{len(shift_results)}")
if with_data == len(shift_results):
    print("=> Shift data appears available for all tested eras.")
elif with_data > 0:
    empty_seasons = [r[1] for r in shift_results if r[3] == 0]
    print(f"=> Some eras missing shift data: {empty_seasons}")
else:
    print("=> No shift data returned. API may have changed or games not found.")

## 8. Summary and gate decision

**Interpret the results above as an exploratory statistical screen, not the final model-design gate.**

> Does `faceoff_zone_code` alone capture sufficient zone-deployment context, or is full shift data ingestion needed?

**Screening criteria:**

1. **Zone signal exists (section 2):** The all-shot chi-squared test should be significant, at least one zone should have an adequately powered practical effect (|Cohen's h| >= 0.2), and the rate estimates should include 95% CIs.

2. **Signal is time-bounded (section 3):** Early windows should show stronger practical effects than later windows, with underpowered cells explicitly ignored.

3. **Steady-state shots differ (section 4):** The phase comparison should show a statistically and practically meaningful difference between immediate post-faceoff and steady-state shots.

4. **Interaction adds value (section 5):** The zone-recency interaction is only a candidate if the time-window tests show powered practical effects.

5. **Shift data available (section 7):** Historical availability informs feasibility if shift ingestion is later deemed necessary.

**Final gate:** Do not add `faceoff_zone_code`, `seconds_since_faceoff`, or the interaction feature to the production xG model based on this notebook alone. The feature decision requires the temporal ablation and leakage audit in `model_validation_framework.ipynb`; full-dataset patterns here are exploratory until they improve held-out performance.


In [ ]:
conn.close()
print("Done.")